In [6]:
import pandas as pd
import numpy as np

application = pd.read_csv(
    "../data/raw/application_train.csv"
)

bureau = pd.read_csv(
    "../data/raw/bureau.csv"
)

print(application.shape)
print(bureau.shape)

(307511, 122)
(1716428, 17)


In [7]:
bureau_features = bureau.groupby(
    "SK_ID_CURR"
).agg(
    bureau_count=("SK_ID_BUREAU","count"),

    active_loans=(
        "CREDIT_ACTIVE",
        lambda x: (x=="Active").sum()
    ),

    closed_loans=(
        "CREDIT_ACTIVE",
        lambda x: (x=="Closed").sum()
    ),

    bureau_credit_mean=(
        "AMT_CREDIT_SUM",
        "mean"
    ),

    total_credit=(
        "AMT_CREDIT_SUM",
        "sum"
    ),

    avg_debt=(
        "AMT_CREDIT_SUM_DEBT",
        "mean"
    ),

    max_debt=(
        "AMT_CREDIT_SUM_DEBT",
        "max"
    ),

    bureau_overdue_max=(
        "AMT_CREDIT_MAX_OVERDUE",
        "max"
    )
).reset_index()

In [9]:
bureau_features[
    "debt_credit_ratio"
] = (
    bureau_features["avg_debt"]
    /
    bureau_features["bureau_credit_mean"]
)

bureau_features[
    "active_loan_ratio"
] = (
    bureau_features["active_loans"]
    /
    bureau_features["bureau_count"]
)

bureau_features[
    "closed_loan_ratio"
] = (
    bureau_features["closed_loans"]
    /
    bureau_features["bureau_count"]
)

In [10]:
bureau_features = bureau_features.replace(
    [np.inf,-np.inf],
    np.nan
)

In [11]:
application_bureau = application.merge(
    bureau_features,
    on="SK_ID_CURR",
    how="left"
)

In [13]:
application_bureau.to_csv(
    "../data/processed/application_bureau.csv",
    index=False
)

In [14]:
print(application_bureau.shape)

bureau_cols = [
    c
    for c in application_bureau.columns
    if c in bureau_features.columns
]

print(bureau_cols)

(307511, 133)
['SK_ID_CURR', 'bureau_count', 'active_loans', 'closed_loans', 'bureau_credit_mean', 'total_credit', 'avg_debt', 'max_debt', 'bureau_overdue_max', 'debt_credit_ratio', 'active_loan_ratio', 'closed_loan_ratio']
